<a href="https://colab.research.google.com/github/SilvanaTomsig/COMISION-90545-DATA-SCIENCE-II/blob/main/Unidad4_Practica_NuevosDatasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏋️ Unidad 4 — SQL para Ciencia de Datos
## Notebook de Práctica: Nuevos Datasets

Este notebook tiene **3 datasets completamente nuevos** para que practiques sin ver las respuestas del notebook anterior.

Cada sección está organizada así:
1. 📋 **Contexto** — qué datos tenés disponibles
2. 🎯 **Consigna** — qué tenés que hacer
3. 📝 **Tu espacio** — celda en blanco para que escribas
4. ✅ **Solución** — la respuesta correcta (¡intentalo antes de mirar!)

---
**Bases de datos:**
- 🎬 **Dataset A** — `cine`: Películas, directores, actores y proyecciones  
- 🏥 **Dataset B** — `clinica`: Pacientes, médicos, turnos y diagnósticos  
- 🛒 **Dataset C** — `ecommerce`: Productos, usuarios, carritos y reseñas  


## ⚙️ Setup inicial

In [ ]:
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
    import duckdb

import sqlite3
import pandas as pd
print("✅ Listo.")


✅ Listo.


---
# 🎬 Dataset A — Sistema de Cine

### Estructura de la base de datos

```
Directores(id, nombre, nacionalidad, anio_nacimiento)
Peliculas(id, titulo, anio, genero, duracion_min, director_id, presupuesto_musd)
Actores(id, nombre, nacionalidad)
Casting(pelicula_id, actor_id, personaje, protagonista BOOLEAN)
Salas(id, nombre, capacidad, ciudad)
Proyecciones(id, pelicula_id, sala_id, fecha, hora, precio)
```


### Crear y poblar la base de datos Cine

In [ ]:
import sqlite3

cine = sqlite3.connect(":memory:")

cine.executescript("""
CREATE TABLE Directores (
    id               INTEGER PRIMARY KEY,
    nombre           TEXT,
    nacionalidad     TEXT,
    anio_nacimiento  INTEGER
);

CREATE TABLE Peliculas (
    id              INTEGER PRIMARY KEY,
    titulo          TEXT,
    anio            INTEGER,
    genero          TEXT,
    duracion_min    INTEGER,
    director_id     INTEGER REFERENCES Directores(id),
    presupuesto_musd REAL
);

CREATE TABLE Actores (
    id           INTEGER PRIMARY KEY,
    nombre       TEXT,
    nacionalidad TEXT
);

CREATE TABLE Casting (
    pelicula_id  INTEGER REFERENCES Peliculas(id),
    actor_id     INTEGER REFERENCES Actores(id),
    personaje    TEXT,
    protagonista INTEGER   -- 1 = True, 0 = False
);

CREATE TABLE Salas (
    id        INTEGER PRIMARY KEY,
    nombre    TEXT,
    capacidad INTEGER,
    ciudad    TEXT
);

CREATE TABLE Proyecciones (
    id          INTEGER PRIMARY KEY,
    pelicula_id INTEGER REFERENCES Peliculas(id),
    sala_id     INTEGER REFERENCES Salas(id),
    fecha       TEXT,
    hora        TEXT,
    precio      REAL
);
""")

cine.executemany("INSERT INTO Directores VALUES (?,?,?,?)", [
    (1,"Christopher Nolan",   "Británico",  1970),
    (2,"Guillermo del Toro",  "Mexicano",   1964),
    (3,"Greta Gerwig",        "Estadounidense",1983),
    (4,"Bong Joon-ho",        "Coreano",    1969),
    (5,"Denis Villeneuve",    "Canadiense", 1967),
])

cine.executemany("INSERT INTO Peliculas VALUES (?,?,?,?,?,?,?)", [
    (1,"Inception",             2010,"Ciencia Ficción",148,1,160.0),
    (2,"Interstellar",          2014,"Ciencia Ficción",169,1,165.0),
    (3,"Tenet",                 2020,"Acción",         150,1,205.0),
    (4,"La forma del agua",     2017,"Romance/Terror", 123,2, 19.4),
    (5,"Barbie",                2023,"Comedia",        114,3,145.0),
    (6,"Lady Bird",             2017,"Drama",           93,3, 10.0),
    (7,"Parasite",              2019,"Thriller",       132,4, 11.4),
    (8,"Memories of Murder",    2003,"Thriller",       131,4,  2.8),
    (9,"Dune: Part One",        2021,"Ciencia Ficción",155,5,165.0),
    (10,"Arrival",              2016,"Ciencia Ficción",116,5, 47.0),
])

cine.executemany("INSERT INTO Actores VALUES (?,?,?)", [
    (1,"Leonardo DiCaprio","Estadounidense"),
    (2,"Tom Hardy",         "Británico"),
    (3,"Margot Robbie",     "Australiana"),
    (4,"Ryan Gosling",      "Canadiense"),
    (5,"Timothée Chalamet", "Estadounidense"),
    (6,"Saoirse Ronan",     "Irlandesa"),
    (7,"Song Kang-ho",      "Coreano"),
    (8,"Amy Adams",         "Estadounidense"),
    (9,"Matthew McConaughey","Estadounidense"),
    (10,"Sally Hawkins",    "Británica"),
])

cine.executemany("INSERT INTO Casting VALUES (?,?,?,?)", [
    (1,1,"Dom Cobb",1),(1,2,"Eames",0),
    (2,9,"Cooper",1),(2,1,"Cooper alt",0),
    (3,2,"Protagonist",1),
    (4,10,"Elisa",1),
    (5,3,"Barbie",1),(5,4,"Ken",1),
    (6,6,"Christine",1),
    (7,7,"Ki-taek",1),
    (9,5,"Paul Atreides",1),
    (10,8,"Louise Banks",1),
])

cine.executemany("INSERT INTO Salas VALUES (?,?,?,?)", [
    (1,"Gran Pantalla", 250,"Buenos Aires"),
    (2,"Sala Premium",  80, "Buenos Aires"),
    (3,"Cinearte",      120,"Córdoba"),
    (4,"Multiplex 1",   200,"Rosario"),
    (5,"IMAX Palermo",  300,"Buenos Aires"),
])

cine.executemany("INSERT INTO Proyecciones VALUES (?,?,?,?,?,?)", [
    (1,  1,1,"2024-06-01","19:00",1200.0),
    (2,  1,5,"2024-06-01","21:30",1800.0),
    (3,  2,5,"2024-06-02","20:00",1800.0),
    (4,  5,1,"2024-06-02","18:00",1100.0),
    (5,  5,2,"2024-06-02","20:30",1400.0),
    (6,  7,3,"2024-06-03","19:30",1000.0),
    (7,  9,5,"2024-06-03","21:00",2000.0),
    (8,  9,4,"2024-06-03","19:00",1300.0),
    (9,  3,1,"2024-06-04","22:00",1200.0),
    (10, 6,3,"2024-06-04","17:00",900.0),
    (11, 4,2,"2024-06-05","20:00",1400.0),
    (12,10,4,"2024-06-05","18:30",1100.0),
])

cine.commit()
print("✅ Base de datos CINE lista. Tablas:",
      [r[0] for r in cine.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])

✅ Base de datos CINE lista. Tablas: ['Directores', 'Peliculas', 'Actores', 'Casting', 'Salas', 'Proyecciones']


---
### 🎯 Consigna A1 — Películas por género y director

Mostrá el **título, año, duración y nombre del director** de todas las películas  
de género **Ciencia Ficción**, ordenadas por año de estreno (más reciente primero).


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


In [ ]:
pd.read_sql_query("""
    SELECT
        p.titulo,
        p.anio,
        p.duracion_min  AS duracion,
        d.nombre        AS director
    FROM   Peliculas p
    JOIN   Directores d ON p.director_id = d.id
    WHERE  p.genero = 'Ciencia Ficción'
    ORDER  BY p.anio DESC
""", cine)


,titulo,anio,duracion,director
0,Dune: Part One,2021,155,Denis Villeneuve
1,Arrival,2016,116,Denis Villeneuve
2,Interstellar,2014,169,Christopher Nolan
3,Inception,2010,148,Christopher Nolan


<details>
<summary>✅ Ver solución A1</summary>

```python
pd.read_sql_query("""
    SELECT
        p.titulo,
        p.anio,
        p.duracion_min  AS duracion,
        d.nombre        AS director
    FROM   Peliculas p
    JOIN   Directores d ON p.director_id = d.id
    WHERE  p.genero = 'Ciencia Ficción'
    ORDER  BY p.anio DESC
""", cine)
```
</details>


### 🎯 Consigna A2 — Directores más prolíficos

¿Cuántas películas tiene cada director en la base?  
Mostrá **nombre del director, cantidad de películas y presupuesto promedio**  
(redondeado a 1 decimal), ordenado por cantidad de películas descendente.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución A2</summary>

```python
pd.read_sql_query("""
    SELECT
        d.nombre           AS director,
        COUNT(*)           AS peliculas,
        ROUND(AVG(p.presupuesto_musd), 1) AS presupuesto_prom_musd
    FROM   Directores d
    JOIN   Peliculas  p ON d.id = p.director_id
    GROUP  BY d.nombre
    ORDER  BY peliculas DESC
""", cine)
```
</details>


### 🎯 Consigna A3 — Ingreso potencial por proyección

Calculá el **ingreso máximo posible** de cada proyección (precio × capacidad de sala).  
Mostrá: fecha, hora, título de la película, nombre de sala, ciudad, precio y  
la columna calculada `ingreso_maximo`. Ordená por ingreso de mayor a menor.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución A3</summary>

```python
pd.read_sql_query("""
    SELECT
        pr.fecha,
        pr.hora,
        p.titulo          AS pelicula,
        s.nombre          AS sala,
        s.ciudad,
        pr.precio,
        ROUND(pr.precio * s.capacidad, 0) AS ingreso_maximo
    FROM   Proyecciones pr
    JOIN   Peliculas    p  ON pr.pelicula_id = p.id
    JOIN   Salas        s  ON pr.sala_id     = s.id
    ORDER  BY ingreso_maximo DESC
""", cine)
```
</details>


### 🎯 Consigna A4 — DML: Insertar una nueva proyección

Insertá una nueva proyección de la película **"Parasite"** (id=7)  
en la **Sala Premium** (id=2), el **2024-06-10** a las **18:00**,  
con un precio de **$1.300**.  
Después verificá que se insertó correctamente consultando todas las proyecciones de esa película.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución A4</summary>

```python
cur_cine = cine.cursor()
cur_cine.execute(
    "INSERT INTO Proyecciones (pelicula_id, sala_id, fecha, hora, precio) VALUES (?,?,?,?,?)",
    (7, 2, "2024-06-10", "18:00", 1300.0)
)
cine.commit()
print(f"✅ Proyección insertada con ID {cur_cine.lastrowid}")

pd.read_sql_query("""
    SELECT pr.*, p.titulo, s.nombre AS sala
    FROM   Proyecciones pr
    JOIN   Peliculas p ON pr.pelicula_id = p.id
    JOIN   Salas     s ON pr.sala_id     = s.id
    WHERE  p.titulo = 'Parasite'
""", cine)
```
</details>


---
# 🏥 Dataset B — Sistema de Clínica

### Estructura de la base de datos

```
Medicos(id, nombre, especialidad, anios_experiencia)
Pacientes(id, nombre, fecha_nacimiento, obra_social)
Turnos(id, paciente_id, medico_id, fecha, hora, estado)
Diagnosticos(id, turno_id, cie10_codigo, descripcion, tratamiento)
Medicamentos(id, nombre, laboratorio, precio)
Recetas(id, diagnostico_id, medicamento_id, dosis, duracion_dias)
```


### Crear y poblar la base de datos Clínica

In [ ]:
clinica = sqlite3.connect(":memory:")

clinica.executescript("""
CREATE TABLE Medicos (
    id                INTEGER PRIMARY KEY,
    nombre            TEXT,
    especialidad      TEXT,
    anios_experiencia INTEGER
);

CREATE TABLE Pacientes (
    id                INTEGER PRIMARY KEY,
    nombre            TEXT,
    fecha_nacimiento  TEXT,
    obra_social       TEXT
);

CREATE TABLE Turnos (
    id          INTEGER PRIMARY KEY,
    paciente_id INTEGER REFERENCES Pacientes(id),
    medico_id   INTEGER REFERENCES Medicos(id),
    fecha       TEXT,
    hora        TEXT,
    estado      TEXT   -- 'Realizado', 'Cancelado', 'Pendiente'
);

CREATE TABLE Diagnosticos (
    id          INTEGER PRIMARY KEY,
    turno_id    INTEGER REFERENCES Turnos(id),
    cie10_codigo TEXT,
    descripcion TEXT,
    tratamiento TEXT
);

CREATE TABLE Medicamentos (
    id           INTEGER PRIMARY KEY,
    nombre       TEXT,
    laboratorio  TEXT,
    precio       REAL
);

CREATE TABLE Recetas (
    id             INTEGER PRIMARY KEY,
    diagnostico_id INTEGER REFERENCES Diagnosticos(id),
    medicamento_id INTEGER REFERENCES Medicamentos(id),
    dosis          TEXT,
    duracion_dias  INTEGER
);
""")

clinica.executemany("INSERT INTO Medicos VALUES(?,?,?,?)", [
    (1,"Dr. Pablo Suárez",    "Cardiología",     18),
    (2,"Dra. Elena Vidal",    "Clínica Médica",  12),
    (3,"Dr. Lucas Herrera",   "Traumatología",    8),
    (4,"Dra. Marta Flores",   "Pediatría",       15),
    (5,"Dr. Rodrigo Paz",     "Neurología",      20),
])

clinica.executemany("INSERT INTO Pacientes VALUES(?,?,?,?)", [
    (1,"Fernando Aguirre",  "1980-03-12","OSDE"),
    (2,"Natalia Benítez",   "1995-07-28","Swiss Medical"),
    (3,"Tomás Castillo",    "1965-11-05","IOMA"),
    (4,"Gabriela Romero",   "2005-02-14","PAMI"),
    (5,"Diego Ferreira",    "1978-09-30","Galeno"),
    (6,"Camila Ortega",     "2018-04-22","OSDE"),
    (7,"Hernán Molina",     "1955-06-01","PAMI"),
])

clinica.executemany("INSERT INTO Turnos VALUES(?,?,?,?,?,?)", [
    (1, 1,1,"2024-05-10","09:00","Realizado"),
    (2, 2,2,"2024-05-10","10:30","Realizado"),
    (3, 3,3,"2024-05-11","08:00","Realizado"),
    (4, 4,4,"2024-05-11","11:00","Realizado"),
    (5, 5,5,"2024-05-12","09:30","Realizado"),
    (6, 6,4,"2024-05-12","10:00","Realizado"),
    (7, 7,1,"2024-05-13","08:30","Realizado"),
    (8, 1,5,"2024-05-14","09:00","Cancelado"),
    (9, 2,1,"2024-05-15","10:00","Pendiente"),
    (10,3,2,"2024-05-15","11:30","Pendiente"),
])

clinica.executemany("INSERT INTO Diagnosticos VALUES(?,?,?,?,?)", [
    (1,1,"I10",  "Hipertensión arterial",   "Cambio de estilo de vida + medicación"),
    (2,2,"J06.9","Infección respiratoria",  "Reposo + antibióticos"),
    (3,3,"M79.3","Dolor en rodilla derecha","Kinesiología + antiinflamatorio"),
    (4,4,"J45",  "Asma bronquial",          "Broncodilatador"),
    (5,5,"G43",  "Migraña crónica",         "Profilaxis con triptanos"),
    (6,6,"J06.9","Faringitis viral",        "Reposo + paracetamol"),
    (7,7,"I25",  "Cardiopatía isquémica",   "Medicación + seguimiento"),
])

clinica.executemany("INSERT INTO Medicamentos VALUES(?,?,?,?)", [
    (1,"Enalapril 10mg",  "Roemmers",  850.0),
    (2,"Amoxicilina 500mg","Bago",     1200.0),
    (3,"Ibuprofeno 400mg", "Gador",    600.0),
    (4,"Salbutamol inh.",  "GSK",     2400.0),
    (5,"Sumatriptán 50mg", "Novartis",3800.0),
    (6,"Paracetamol 500mg","Roemmers", 450.0),
    (7,"AAS 100mg",        "Bayer",    550.0),
])

clinica.executemany("INSERT INTO Recetas VALUES(?,?,?,?,?)", [
    (1,1,1,"1 comp. cada 12h",90),
    (2,2,2,"1 comp. cada 8h", 7),
    (3,3,3,"1 comp. cada 8h",10),
    (4,4,4,"2 puffs cada 4h", 30),
    (5,5,5,"1 comp. al inicio de crisis",60),
    (6,6,6,"1 comp. cada 8h si dolor",5),
    (7,7,1,"1 comp. cada 12h",180),
    (8,7,7,"1 comp. diaria",   365),
])

clinica.commit()
print("✅ Base de datos CLÍNICA lista.")


---
### 🎯 Consigna B1 — Turnos realizados por médico

¿Cuántos turnos **Realizados** tiene cada médico?  
Mostrá nombre del médico, especialidad y cantidad de turnos,  
ordenado de mayor a menor.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución B1</summary>

```python
pd.read_sql_query("""
    SELECT
        m.nombre       AS medico,
        m.especialidad,
        COUNT(*)       AS turnos_realizados
    FROM   Medicos m
    JOIN   Turnos  t ON m.id = t.medico_id
    WHERE  t.estado = 'Realizado'
    GROUP  BY m.nombre, m.especialidad
    ORDER  BY turnos_realizados DESC
""", clinica)
```
</details>


### 🎯 Consigna B2 — Costo de recetas por paciente

Calculá el **costo total de los medicamentos recetados** a cada paciente  
(precio × duración_días).  
Mostrá nombre del paciente, obra social y costo total, ordenado por costo descendente.

> Pista: necesitás hacer JOIN entre Pacientes → Turnos → Diagnosticos → Recetas → Medicamentos


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución B2</summary>

```python
pd.read_sql_query("""
    SELECT
        p.nombre       AS paciente,
        p.obra_social,
        ROUND(SUM(med.precio * r.duracion_dias), 2) AS costo_total_recetas
    FROM   Pacientes    p
    JOIN   Turnos       t   ON p.id        = t.paciente_id
    JOIN   Diagnosticos d   ON t.id        = d.turno_id
    JOIN   Recetas      r   ON d.id        = r.diagnostico_id
    JOIN   Medicamentos med ON r.medicamento_id = med.id
    GROUP  BY p.nombre, p.obra_social
    ORDER  BY costo_total_recetas DESC
""", clinica)
```
</details>


### 🎯 Consigna B3 — Pacientes sin turno pendiente

Encontrá todos los pacientes que **NO tienen ningún turno en estado 'Pendiente'**.  
Usá una subconsulta con `NOT IN` o un `LEFT JOIN`.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución B3 (dos variantes)</summary>

```python
# Variante 1 — NOT IN
pd.read_sql_query("""
    SELECT nombre, obra_social
    FROM   Pacientes
    WHERE  id NOT IN (
        SELECT DISTINCT paciente_id
        FROM   Turnos
        WHERE  estado = 'Pendiente'
    )
""", clinica)

# Variante 2 — LEFT JOIN + IS NULL
pd.read_sql_query("""
    SELECT p.nombre, p.obra_social
    FROM   Pacientes p
    LEFT JOIN Turnos t ON p.id = t.paciente_id AND t.estado = 'Pendiente'
    WHERE  t.id IS NULL
""", clinica)
```
</details>


### 🎯 Consigna B4 — UPDATE: Cancelar turno

El paciente con id=2 canceló el turno pendiente (id=9).  
Actualizá su estado a `'Cancelado'` y verificá el cambio.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución B4</summary>

```python
cur_cli = clinica.cursor()
cur_cli.execute(
    "UPDATE Turnos SET estado = 'Cancelado' WHERE id = 9 AND paciente_id = 2"
)
clinica.commit()
print(f"✅ {cur_cli.rowcount} turno(s) actualizado(s).")

pd.read_sql_query("SELECT * FROM Turnos WHERE paciente_id = 2", clinica)
```
</details>


---
# 🛒 Dataset C — E-Commerce con DuckDB

Este dataset trabaja directamente con **DataFrames de Pandas** usando **DuckDB**,  
sin necesitar una base SQLite previa.

### Tablas disponibles (como DataFrames)

```
df_usuarios     — id, nombre, ciudad, fecha_registro, nivel (Bronze/Silver/Gold)
df_productos    — id, nombre, categoria, precio, stock
df_pedidos      — id, usuario_id, fecha, estado, descuento_pct
df_items        — pedido_id, producto_id, cantidad, precio_unitario
df_resenias     — id, producto_id, usuario_id, puntuacion (1-5), comentario
```


### Crear los DataFrames del E-Commerce

In [ ]:
import random
from datetime import date, timedelta
random.seed(99)

# ── Usuarios ──────────────────────────────────────────────
df_usuarios = pd.DataFrame([
    (1,"Valentina R.","Buenos Aires","2023-01-15","Gold"),
    (2,"Mateo G.",    "Córdoba",     "2023-03-22","Silver"),
    (3,"Lucía M.",    "Rosario",     "2022-11-10","Bronze"),
    (4,"Sebastián F.","Buenos Aires","2024-01-08","Silver"),
    (5,"Isabella P.", "Mendoza",     "2023-07-19","Bronze"),
    (6,"Agustín H.",  "Buenos Aires","2022-08-30","Gold"),
    (7,"Martina L.",  "Córdoba",     "2024-02-14","Bronze"),
    (8,"Tomás V.",    "Rosario",     "2023-05-05","Silver"),
], columns=["id","nombre","ciudad","fecha_registro","nivel"])

# ── Productos ─────────────────────────────────────────────
df_productos = pd.DataFrame([
    (1,"Smartphone X12", "Electrónica",  89999,  45),
    (2,"Auriculares BT", "Electrónica",   8499, 120),
    (3,"Zapatillas Run", "Indumentaria", 24990,  60),
    (4,"Mochila Urbana", "Accesorios",   12500,  80),
    (5,"Cámara DSLR",    "Electrónica", 150000,  15),
    (6,"Remera Básica",  "Indumentaria",  4999, 200),
    (7,"Notebook UltraBook","Electrónica",349990, 20),
    (8,"Taza Térmica",   "Hogar",         3499, 150),
    (9,"Libro: Python DS","Libros",       5990, 300),
    (10,"Smart TV 55''", "Electrónica",  189999,  30),
], columns=["id","nombre","categoria","precio","stock"])

# ── Pedidos ───────────────────────────────────────────────
pedidos = []
random.seed(99)
start = date(2024,1,1)
for pid in range(1,31):
    uid   = random.randint(1,8)
    dias  = random.randint(0,180)
    fecha = str(start + timedelta(days=dias))
    estado= random.choice(["Entregado","Entregado","Entregado","Cancelado","Enviado"])
    desc  = random.choice([0,0,0,5,10,15])
    pedidos.append((pid,uid,fecha,estado,desc))
df_pedidos = pd.DataFrame(pedidos, columns=["id","usuario_id","fecha","estado","descuento_pct"])

# ── Items ─────────────────────────────────────────────────
items = []
iid = 1
random.seed(77)
for _, ped in df_pedidos.iterrows():
    n_items = random.randint(1,3)
    prods_usados = random.sample(range(1,11), min(n_items,10))
    for prod_id in prods_usados:
        cant = random.randint(1,3)
        precio = df_productos.loc[df_productos.id==prod_id,"precio"].values[0]
        items.append((ped["id"], prod_id, cant, float(precio)))
        iid += 1
df_items = pd.DataFrame(items, columns=["pedido_id","producto_id","cantidad","precio_unitario"])

# ── Reseñas ───────────────────────────────────────────────
comentarios = [
    "Excelente producto","Muy buena calidad","Llegó bien embalado",
    "Justo lo que necesitaba","Podría ser mejor","Demasiado caro para lo que es",
    "Lo recomiendo","Buena relación precio-calidad","No era lo que esperaba","Perfecto"
]
random.seed(55)
resenias = []
for i in range(1,26):
    prod_id = random.randint(1,10)
    user_id = random.randint(1,8)
    punt    = random.randint(1,5)
    coment  = random.choice(comentarios)
    resenias.append((i, prod_id, user_id, punt, coment))
df_resenias = pd.DataFrame(resenias, columns=["id","producto_id","usuario_id","puntuacion","comentario"])

print("✅ DataFrames listos:")
print(f"  usuarios: {len(df_usuarios)} | productos: {len(df_productos)}")
print(f"  pedidos:  {len(df_pedidos)}  | items: {len(df_items)} | reseñas: {len(df_resenias)}")


---
### 🎯 Consigna C1 — Facturación por categoría

Calculá el **total facturado** por cada categoría de productos.  
Considerá el descuento aplicado al pedido: `total = cantidad × precio × (1 - descuento_pct/100)`.  
Mostrá solo pedidos con estado `'Entregado'`.  
Ordená de mayor a menor facturación.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución C1</summary>

```python
duckdb.query("""
    SELECT
        p.categoria,
        ROUND(SUM(i.cantidad * i.precio_unitario * (1 - ped.descuento_pct / 100.0)), 2)
            AS facturacion_total
    FROM   df_items    i
    JOIN   df_pedidos  ped ON i.pedido_id   = ped.id
    JOIN   df_productos p  ON i.producto_id = p.id
    WHERE  ped.estado = 'Entregado'
    GROUP  BY p.categoria
    ORDER  BY facturacion_total DESC
""").df()
```
</details>


### 🎯 Consigna C2 — Usuarios más valiosos

¿Cuáles son los **3 usuarios con mayor gasto total** (pedidos Entregados)?  
Mostrá nombre, ciudad, nivel y el total gastado.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución C2</summary>

```python
duckdb.query("""
    SELECT
        u.nombre,
        u.ciudad,
        u.nivel,
        ROUND(SUM(i.cantidad * i.precio_unitario * (1 - ped.descuento_pct/100.0)), 2)
            AS gasto_total
    FROM   df_usuarios  u
    JOIN   df_pedidos   ped ON u.id          = ped.usuario_id
    JOIN   df_items     i   ON ped.id        = i.pedido_id
    WHERE  ped.estado = 'Entregado'
    GROUP  BY u.nombre, u.ciudad, u.nivel
    ORDER  BY gasto_total DESC
    LIMIT  3
""").df()
```
</details>


### 🎯 Consigna C3 — Rating promedio y ventas por producto

Para cada producto, calculá:
- `puntuacion_promedio` (de las reseñas)
- `unidades_vendidas` (de los items de pedidos Entregados)
- `ingreso_total`

Mostrá solo productos con al menos **1 reseña**, ordenados por puntuación promedio descendente.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución C3</summary>

```python
duckdb.query("""
    SELECT
        p.nombre                         AS producto,
        p.categoria,
        ROUND(AVG(r.puntuacion), 2)      AS puntuacion_promedio,
        COALESCE(SUM(i.cantidad), 0)     AS unidades_vendidas,
        ROUND(COALESCE(SUM(i.cantidad * i.precio_unitario), 0), 2) AS ingreso_total
    FROM   df_productos p
    JOIN   df_resenias  r   ON p.id = r.producto_id
    LEFT JOIN df_items  i   ON p.id = i.producto_id
    LEFT JOIN df_pedidos ped ON i.pedido_id = ped.id AND ped.estado = 'Entregado'
    GROUP  BY p.id, p.nombre, p.categoria
    HAVING COUNT(r.id) >= 1
    ORDER  BY puntuacion_promedio DESC
""").df()
```
</details>


### 🎯 Consigna C4 — Tendencia mensual de ventas

Mostrá el **total facturado por mes** (pedidos Entregados) usando  
`strftime('%Y-%m', fecha)` para extraer el mes.  
Incluí también la cantidad de pedidos por mes.  
Ordená cronológicamente.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución C4</summary>

```python
duckdb.query("""
    SELECT
        strftime(ped.fecha, '%Y-%m')   AS mes,
        COUNT(DISTINCT ped.id)         AS cantidad_pedidos,
        ROUND(SUM(i.cantidad * i.precio_unitario * (1 - ped.descuento_pct/100.0)), 2)
            AS facturacion
    FROM   df_pedidos  ped
    JOIN   df_items    i ON ped.id = i.pedido_id
    WHERE  ped.estado = 'Entregado'
    GROUP  BY mes
    ORDER  BY mes
""").df()
```
</details>


### 🎯 Consigna C5 — Productos con bajo stock y alta demanda 🔥

Identificá productos que tienen **stock menor a 50** Y fueron vendidos  
**más de 5 unidades en total** (en pedidos Entregados).  
Estos son los productos en riesgo de ruptura de stock.


In [ ]:
# ✍️ TU CÓDIGO AQUÍ


<details>
<summary>✅ Ver solución C5</summary>

```python
duckdb.query("""
    SELECT
        p.nombre,
        p.categoria,
        p.stock,
        SUM(i.cantidad) AS unidades_vendidas,
        p.stock - SUM(i.cantidad) AS stock_restante_estimado
    FROM   df_productos p
    JOIN   df_items     i   ON p.id = i.producto_id
    JOIN   df_pedidos   ped ON i.pedido_id = ped.id
    WHERE  ped.estado = 'Entregado'
      AND  p.stock < 50
    GROUP  BY p.id, p.nombre, p.categoria, p.stock
    HAVING SUM(i.cantidad) > 5
    ORDER  BY stock_restante_estimado ASC
""").df()
```
</details>


---
## 🎓 ¡Felicitaciones por llegar hasta acá!

### Lo que practicaste en este notebook:

| Dataset | Habilidades trabajadas |
|---------|------------------------|
| 🎬 Cine | SELECT + JOIN, GROUP BY, cálculos, INSERT |
| 🏥 Clínica | JOINs múltiples, subconsultas, NOT IN, UPDATE |
| 🛒 E-Commerce | DuckDB sobre DataFrames, HAVING, tendencias, casos reales |

### Desafíos extra (opcional)
- 🎬 ¿Qué actor/actriz participó en más películas? (Casting + GROUP BY)
- 🏥 ¿Qué obra social tiene mayor costo promedio de medicación por paciente?
- 🛒 ¿Cuál es el ticket promedio por nivel de usuario (Bronze/Silver/Gold)?

---
*¡Seguí practicando! La fluidez en SQL se logra escribiendo consultas todos los días.*
